# lambda_functions
**Prerequisites:** function_basics, parameters_and_arguments, return_values, scope

**Outcome:** After this notebook you will know exactly what a lambda is, how it differs from a regular function at the bytecode level, every legitimate use case, and the clear line between when lambda makes code cleaner and when it makes it unreadable.

## The Problem

In [ ]:
# sorting a list of records by a field
# without lambda you write a whole function for a one-liner operation

jobs = [
    {"id": "job_3", "duration_ms": 300},
    {"id": "job_1", "duration_ms": 100},
    {"id": "job_2", "duration_ms": 200},
]

def get_duration(job):
    return job["duration_ms"]

sorted_jobs = sorted(jobs, key=get_duration)
print(sorted_jobs)

# get_duration is a throwaway function that will never be used again
# it pollutes the namespace and forces the reader to scroll away from the sort call

## The Concept

A lambda is an anonymous function defined with a single expression. It has no name, no statements, and no return keyword — the expression result is always returned automatically. Lambda is syntactic sugar for a `def` that creates a function object inline at the point of use. The result is exactly the same type of object as any function created with `def`. The only difference is that lambda is restricted to a single expression, which makes it unsuitable for anything complex and ideal for short, throwaway operations passed as arguments.

## Minimal Example

In [ ]:
# def version
def double(x):
    return x * 2

# lambda version — identical behaviour
double = lambda x: x * 2

print(double(5))    # 10
print(type(double)) # <class 'function'>

## How Python Does It

Python compiles a lambda expression to the same bytecode as a `def` with a single return statement. The resulting object is a normal function object stored under the name `<lambda>` instead of a real name. It has `__code__`, `__defaults__`, and `__closure__` just like any other function. The restriction to one expression is enforced by the parser, not the bytecode engine — there is nothing you can do in a lambda that you could not do with a `def`, except write multiple statements.

In [ ]:
import dis

def double_def(x):
    return x * 2

double_lambda = lambda x: x * 2

print("--- def ---")
dis.dis(double_def)
print("--- lambda ---")
dis.dis(double_lambda)
# bytecode is identical — lambda is not faster or slower

## Building Up

In [ ]:
# lambda syntax forms

# no arguments
get_status = lambda: "running"
print(get_status())

# one argument
upper = lambda s: s.upper()
print(upper("api"))

# multiple arguments
add = lambda a, b: a + b
print(add(3, 4))

# default argument
greet = lambda name, prefix="Hello": f"{prefix}, {name}"
print(greet("alice"))
print(greet("bob", "Hi"))

In [ ]:
# primary use case: key argument in sorted()
jobs = [
    {"id": "job_3", "duration_ms": 300},
    {"id": "job_1", "duration_ms": 100},
    {"id": "job_2", "duration_ms": 200},
]

by_duration = sorted(jobs, key=lambda j: j["duration_ms"])
by_id       = sorted(jobs, key=lambda j: j["id"])
by_duration_desc = sorted(jobs, key=lambda j: j["duration_ms"], reverse=True)

print(by_duration)
print(by_id)
print(by_duration_desc)

In [ ]:
# sort by multiple fields using a tuple key
records = [
    {"region": "eu-west",  "latency": 200},
    {"region": "us-east",  "latency": 100},
    {"region": "us-east",  "latency": 80},
    {"region": "eu-west",  "latency": 150},
]

sorted_records = sorted(records, key=lambda r: (r["region"], r["latency"]))
for r in sorted_records:
    print(r)

In [ ]:
# lambda with min() and max()
jobs = [
    {"id": "job_1", "duration_ms": 300},
    {"id": "job_2", "duration_ms": 100},
    {"id": "job_3", "duration_ms": 200},
]

slowest  = max(jobs, key=lambda j: j["duration_ms"])
fastest  = min(jobs, key=lambda j: j["duration_ms"])

print(f"slowest : {slowest['id']} — {slowest['duration_ms']}ms")
print(f"fastest : {fastest['id']} — {fastest['duration_ms']}ms")

In [ ]:
# lambda captures enclosing scope — same cell rules as closures
multiplier = 3
triple = lambda x: x * multiplier
print(triple(10))   # 30

multiplier = 10
print(triple(10))   # 100 — captured by reference, not by value

In [ ]:
# conditional expression inside lambda
classify = lambda v: "slow" if v > 200 else "fast"

latencies = [80, 150, 300, 95, 250]
labels = [classify(v) for v in latencies]
print(labels)

## Where It Breaks

In [ ]:
# lambda cannot contain statements — only expressions
try:
    f = lambda x: if x > 0: return x   # SyntaxError
except SyntaxError as e:
    print(e)

# cannot use assignment, loops, try/except, or multiple lines

In [ ]:
# lambda assigned to a name — defeats the purpose
# linters (flake8, ruff) flag this as E731
process = lambda record: record["id"].upper()

# if you are assigning it to a name, use def instead
def process(record):
    return record["id"].upper()

# def gives a real __name__, better tracebacks, and is more readable
print(process.__name__)   # process — not <lambda>

In [ ]:
# complex logic crammed into a lambda — unreadable
# bad
f = lambda r: r["id"] if r.get("status") == "done" and r.get("retries", 0) <= 3 else None

# good — use def when the logic needs explanation
def get_completed_id(record):
    if record.get("status") == "done" and record.get("retries", 0) <= 3:
        return record["id"]
    return None

print(get_completed_id({"id": "job_1", "status": "done", "retries": 2}))

## The Fix

In [ ]:
# the problem from the top — solved with lambda
jobs = [
    {"id": "job_3", "duration_ms": 300},
    {"id": "job_1", "duration_ms": 100},
    {"id": "job_2", "duration_ms": 200},
]

# clean — throwaway key function stays at the call site
sorted_jobs = sorted(jobs, key=lambda j: j["duration_ms"])
print(sorted_jobs)

## In a Real System

In [ ]:
# a configurable pipeline router that dispatches records
# based on a routing key — the key extractor is passed as a lambda

def route_records(records, key_fn, routes):
    result = {k: [] for k in routes}
    result["unrouted"] = []
    for record in records:
        key = key_fn(record)
        if key in routes:
            result[key].append(record)
        else:
            result["unrouted"].append(record)
    return result

records = [
    {"id": "job_1", "region": "us-east", "status": "done"},
    {"id": "job_2", "region": "eu-west", "status": "failed"},
    {"id": "job_3", "region": "us-east", "status": "done"},
    {"id": "job_4", "region": "ap-south","status": "done"},
]

routed = route_records(
    records,
    key_fn=lambda r: r["region"],
    routes=["us-east", "eu-west"]
)

for region, jobs in routed.items():
    print(f"{region}: {[j['id'] for j in jobs]}")

## Performance

Lambda and `def` produce identical bytecode and have identical runtime performance. There is no speed difference. The only cost consideration is that lambda cannot be pickled by name — since it has no module-level name, multiprocessing and some serialisation libraries cannot handle lambdas. If you need to pass a function to a worker process use a named `def` instead.

## Anti-Pattern

In [ ]:
# anti-pattern: lambda where operator.itemgetter is cleaner
from operator import itemgetter

jobs = [
    {"id": "job_3", "duration_ms": 300},
    {"id": "job_1", "duration_ms": 100},
    {"id": "job_2", "duration_ms": 200},
]

# using lambda
sorted(jobs, key=lambda j: j["duration_ms"])

# using itemgetter — marginally faster, no function call overhead
sorted(jobs, key=itemgetter("duration_ms"))

# for simple field extraction itemgetter is idiomatic
# for computed keys (e.g. two fields, transformation) lambda is cleaner
sorted(jobs, key=lambda j: (j["duration_ms"], j["id"]))  # lambda wins here

## Interview Signals

- What is the difference between a lambda and a function created with `def`?
- Why can a lambda only contain a single expression?
- When should you use `def` instead of `lambda`?
- Why cannot lambdas be used with Python's `multiprocessing` module?

## Exercise

In [ ]:
# implement each transformation using a single lambda
# do not use def — each answer must be one lambda expression

records = [
    {"id": "job_3", "region": "eu-west",  "duration_ms": 300, "status": "done"},
    {"id": "job_1", "region": "us-east",  "duration_ms": 100, "status": "failed"},
    {"id": "job_2", "region": "us-east",  "duration_ms": 200, "status": "done"},
    {"id": "job_4", "region": "eu-west",  "duration_ms": 150, "status": "done"},
]

# 1. sort by duration_ms ascending
sort_by_duration = None   # replace None with a lambda

# 2. sort by region then duration_ms
sort_by_region_then_duration = None

# 3. find the job with the longest duration
get_longest = None

# 4. classify each record: 'fast' if duration_ms < 200 else 'slow'
classify = None


# --- assertions ---
sorted_1 = sorted(records, key=sort_by_duration)
assert [r["id"] for r in sorted_1] == ["job_1", "job_4", "job_2", "job_3"], \
    f"got {[r['id'] for r in sorted_1]}"

sorted_2 = sorted(records, key=sort_by_region_then_duration)
assert [r["id"] for r in sorted_2] == ["job_4", "job_3", "job_1", "job_2"], \
    f"got {[r['id'] for r in sorted_2]}"

assert max(records, key=get_longest)["id"] == "job_3"

assert [classify(r) for r in records] == ["slow", "fast", "slow", "fast"], \
    f"got {[classify(r) for r in records]}"

print("all assertions passed")